In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

BASE            = "/content/drive/MyDrive"
REPO_DIR        = f"{BASE}/ChangeFormer"
LEVIR_PATH      = f"{BASE}/LEVIR-CD"
LEVIR_PLUS_PATH = f"{BASE}/LEVIR-CD-Plus"
CKPT_DIR        = f"{BASE}/ChangeFormer_checkpoints"
VIS_DIR         = f"{BASE}/ChangeFormer_vis"
RESUME_PATH     = f"{BASE}/best_model/best_ckpt.pt"
XBD_PATH        = f"{BASE}/xbd_split"
RESUME_DIR      = f"{BASE}/ChangeFormer_resume"

!pip install -q einops timm scipy


In [3]:
!pip uninstall -y datasets

In [4]:
%cd /content/drive/MyDrive/ChangeFormer_resume
!python -c "import os; print(os.getcwd())"

/content/drive/MyDrive/ChangeFormer_resume
/content/drive/MyDrive/ChangeFormer_resume


In [ ]:
import os


COMBINED_PATH = "/content/drive/MyDrive/COMBINED_CD"  # if using Drive version

for split in ["train", "val", "test"]:
    print(f"\n📝 COMBINED_CD {split}:")

    split_dir = os.path.join(COMBINED_PATH, split)
    list_dir = os.path.join(split_dir, "list")
    os.makedirs(list_dir, exist_ok=True)

    txt_path = os.path.join(list_dir, f"{split}.txt")
    img_dir = os.path.join(split_dir, "A")

    if not os.path.isdir(img_dir):
        print(f"  ⚠️ {split}/A not found:", img_dir)
        continue

    imgs = sorted([
        f for f in os.listdir(img_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff"))
    ])

    with open(txt_path, "w") as f:
        f.write("\n".join(imgs))

    print(f"  ✅ {split}.txt: {len(imgs)} entries")
    print(f"  Saved to: {txt_path}")


📝 COMBINED_CD train:
  ✅ train.txt: 1976 entries
  Saved to: /content/drive/MyDrive/COMBINED_CD/train/list/train.txt

📝 COMBINED_CD val:
  ✅ val.txt: 174 entries
  Saved to: /content/drive/MyDrive/COMBINED_CD/val/list/val.txt

📝 COMBINED_CD test:
  ✅ test.txt: 543 entries
  Saved to: /content/drive/MyDrive/COMBINED_CD/test/list/test.txt


In [6]:
COMBINED_PATH = "/content/drive/MyDrive/COMBINED_CD"
for split in ["train", "val", "test"]:
    txt_path = os.path.join(COMBINED_PATH, split, "list", f"{split}.txt")
    with open(txt_path, "r") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    print(split, len(lines), lines[:3])

train 1976 ['levir_000000.png', 'levir_000001.png', 'levir_000002.png']
val 174 ['levir_000000.png', 'levir_000001.png', 'levir_000002.png']
test 543 ['levir_000000.png', 'levir_000001.png', 'levir_000002.png']


In [7]:
%cd "{RESUME_DIR}"

try:
    !python main_cd.py \
        --img_size 256 \
        --batch_size 64 \
        --data_name COMBINED_CD \
        --lr 0.00002 \
        --max_epochs 60 \
        --patience 15 \""
        --num_workers 4 \
        --net_G ChangeFormerV6 \
        --embed_dim 256 \
        --optimizer adamw \
        --lr_policy linear \
        --loss weighted_ce \
        --multi_scale_train True \
        --multi_scale_infer False \
        --shuffle_AB False \
        --split train \
        --split_val val \
        --checkpoint_root "{CKPT_DIR}" \
        --vis_root "{VIS_DIR}" \
        --gpu_ids 0 \
        --project_name CD_ChangeFormerV6_COMBINED_finetune \
        --resume_path "{RESUME_PATH}" \
        --finetune
finally:
    from google.colab import runtime
    runtime.unassign()

/content/drive/MyDrive/ChangeFormer_resume
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
True
[0]
========== Dataset Info ==========
data_name: COMBINED_CD
root_dir: /content/drive/MyDrive/COMBINED_CD
train split: train
val split: val
label_transform: norm
Train size: 1976
Val size: 174
initialize network with normal
cuda:0
================ (Tue Apr 28 22:21:06 2026) ================
gpu_ids: [0] project_name: CD_ChangeFormerV6_COMBINED_finetune checkpoint_root: /content/drive/MyDrive/ChangeFormer_checkpoints vis_root: /content/drive/MyDrive/ChangeFormer_vis num_workers: 4 dataset: CDDataset data_name: COMBINED_CD batch_size: 64 split: train split_val: val img_size: 256 shuffle_AB: False n_class: 2 embed_dim: 256 pretrain: None resume_path: /content/drive/MyDri